In [1]:
import pandas as pd
import numpy as np

In [2]:
kb_df=pd.read_csv(r"C:\Users\VAIBHAV\Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")
kb_df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:
kb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [5]:
print("Shape:",kb_df.shape)
print("Nulls:",kb_df.isnull().sum())
print(kb_df['category'].value_counts())
print(kb_df['intent'].value_counts().head(20))

Shape: (26872, 5)
Nulls: flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_acc

In [6]:
import json
import re

In [7]:
kb_df['clean_instruction'] = kb_df['instruction'].str.lower()

kb_df['clean_instruction'] = kb_df['clean_instruction'].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x)
)

In [8]:
unwanted_terms = [
    'order number',
    'ordernumber'
]

for term in unwanted_terms:
    kb_df['clean_instruction'] = kb_df['clean_instruction'].str.replace(term, '', regex=False)

In [11]:
def retrieve_response(user_query):

    user_query = user_query.lower()

    for index, row in kb_df.iterrows():

        if any(word in row['clean_instruction']
               for word in user_query.split()):

            return {
                'category': row['category'],
                'intent': row['intent'],
                'response': row['response']
            }

    return {
        'category': 'Unknown',
        'intent': 'Unknown',
        'response': 'No matching support solution found.'
    }

In [12]:
escalation_keywords = [
    'angry',
    'refund',
    'cancel',
    'frustrated',
    'not working'
]

In [13]:
def check_escalation(user_query):

    user_query = user_query.lower()

    for word in escalation_keywords:

        if word in user_query:
            return True

    return False

In [15]:
def support_assistant(user_query):
    retrieval_result = retrieve_response(user_query)
    escalation_required = check_escalation(user_query)

    return {
        'category': retrieval_result['category'],
        'intent': retrieval_result['intent'],
        'response': retrieval_result['response'],
        'escalation_required': escalation_required
    }

## Assistant Workflow V1

This section combines KB retrieval and escalation logic into a single support assistant function. The assistant identifies a relevant support response from the knowledge base and flags cases that may require human escalation.

In [17]:
sample_df = kb_df.sample(1000, random_state=42)

In [31]:
sample_df.to_csv('model_ready_kb.csv', index=False)

## Why TF-IDF Is Used In This Project

The initial version of the support assistant relied on basic keyword matching to retrieve support responses from the knowledge base. While this approach worked for simple queries, it struggled with variations in wording, phrasing, and sentence structure.

To improve retrieval quality, the project uses TF-IDF (Term Frequency–Inverse Document Frequency) vectorization.

TF-IDF converts customer queries and knowledge-base instructions into numerical vector representations based on word importance within the dataset. This allows the system to measure textual similarity between a user query and stored support instructions, even when exact keywords are not identical.

For example:

- User Query: "My internet connection keeps dropping"
- KB Instruction: "Network disconnect issue"

Although the wording differs, TF-IDF helps identify that both queries are contextually related.

This improves:
- support response retrieval accuracy
- handling of varied customer phrasing
- intent matching quality
- scalability of the support assistant workflow

The TF-IDF retrieval layer forms the foundation for the assistant’s intelligent query-matching system before introducing more advanced semantic retrieval methods in future project stages.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
tfidf = TfidfVectorizer()

In [21]:
tfidf_matrix = tfidf.fit_transform(
    kb_df['clean_instruction']
)

In [22]:
tfidf_matrix.shape

(26872, 2862)

## Cosine Similarity Retrieval

After converting support instructions into TF-IDF vectors, cosine similarity is used to measure how closely a user query matches the existing knowledge-base instructions.

Cosine similarity compares the angle between vector representations of the user query and KB instructions. Higher similarity scores indicate stronger contextual similarity between the texts.

This allows the assistant to retrieve relevant support responses even when exact keywords are not identical.

In [23]:
user_query = "My payment failed during checkout"

In [24]:
query_vector = tfidf.transform([user_query])

In [25]:
similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
)

In [28]:
best_match_index = similarity_scores.argmax()
best_match = kb_df.iloc[best_match_index]
print("Category:", best_match['category'])
print("Intent:", best_match['intent'])
print("Response:", best_match['response'])

Category: PAYMENT
Intent: payment_issue
Response: I apologize for the inconvenience caused by the failed payment. It's important to us that you have a smooth and seamless experience. To report the errors you encountered during the payment process, please provide us with more details about the specific issues you faced. This will allow us to investigate and rectify the problem promptly. We appreciate your cooperation and rest assured, our dedicated support team is here to assist you every step of the way. Thank you for bringing this to our attention.


### Testing more queries

In [34]:
user_query = input("Enter a new query:")
query_vector = tfidf.transform([user_query])

similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
)

best_match_index = similarity_scores.argmax()

best_match = kb_df.iloc[best_match_index]

print("Category:", best_match['category'])
print("Intent:", best_match['intent'])
print("Response:", best_match['response'])

Enter a new query: how do I get into my account


Category: REFUND
Intent: get_refund
Response: I acknowledge that you're looking for guidance on how to obtain a rebate. To assist you, I recommend following these steps:

1. Check the terms and conditions: Review the conditions stated in your purchase agreement or any rebate offers provided. Ensure that you meet all the eligibility criteria to receive a rebate.

2. Gather necessary documentation: Collect all the required documents, such as purchase receipts, proof of payment, rebate forms, and any other supporting evidence.

3. Contact customer service: Reach out to our customer service team through our helpline or online chat. Provide them with details about your purchase and express your desire to request a rebate. They will guide you through the specific process for submitting your rebate request.

4. Submit the rebate request: Follow the instructions provided by customer service for submitting your rebate request. Make sure to attach all the required documentation and provide accur

## Retrieval Limitation Observed

A query such as "How do I get into my account?" was incorrectly matched with a refund-related intent because the phrase did not contain strong login-specific keywords. This highlights a limitation of TF-IDF retrieval when user phrasing is indirect or ambiguous.

To improve reliability, query normalization rules and confidence thresholds were added before retrieval.

In [33]:
best_score = similarity_scores[0][best_match_index]

if best_score < 0.35:
    print("Low confidence match. Escalate to human support.")
else:
    print("Category:", best_match['category'])
    print("Intent:", best_match['intent'])
    print("Response:", best_match['response'])
    print("Similarity Score:", best_score)

Category: REFUND
Intent: get_refund
Response: I acknowledge that you're looking for guidance on how to obtain a rebate. To assist you, I recommend following these steps:

1. Check the terms and conditions: Review the conditions stated in your purchase agreement or any rebate offers provided. Ensure that you meet all the eligibility criteria to receive a rebate.

2. Gather necessary documentation: Collect all the required documents, such as purchase receipts, proof of payment, rebate forms, and any other supporting evidence.

3. Contact customer service: Reach out to our customer service team through our helpline or online chat. Provide them with details about your purchase and express your desire to request a rebate. They will guide you through the specific process for submitting your rebate request.

4. Submit the rebate request: Follow the instructions provided by customer service for submitting your rebate request. Make sure to attach all the required documentation and provide accur

In [35]:
def smart_retrieve_response(user_query, top_n=3, threshold=0.45):
    clean_query = user_query.lower()
    clean_query = re.sub(r'[^a-zA-Z0-9\s]', '', clean_query)

    query_vector = tfidf.transform([clean_query])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    top_results = []
    for index in top_indices:
        top_results.append({
            'category': kb_df.iloc[index]['category'],
            'intent': kb_df.iloc[index]['intent'],
            'response': kb_df.iloc[index]['response'],
            'similarity_score': similarity_scores[index]
        })

    best_result = top_results[0]

    if best_result['similarity_score'] < threshold:
        return {
            'category': 'Unknown',
            'intent': 'clarification_required',
            'response': 'I want to make sure I understand correctly. Could you please provide a little more detail about the issue?',
            'similarity_score': best_result['similarity_score'],
            'escalation_required': True,
            'internal_top_matches': top_results
        }

    return {
        'category': best_result['category'],
        'intent': best_result['intent'],
        'response': best_result['response'],
        'similarity_score': best_result['similarity_score'],
        'escalation_required': False,
        'internal_top_matches': top_results
    }

In [36]:
result = smart_retrieve_response("How do I get into my account?")

print(result['response'])

I acknowledge that you're looking for guidance on how to obtain a rebate. To assist you, I recommend following these steps:

1. Check the terms and conditions: Review the conditions stated in your purchase agreement or any rebate offers provided. Ensure that you meet all the eligibility criteria to receive a rebate.

2. Gather necessary documentation: Collect all the required documents, such as purchase receipts, proof of payment, rebate forms, and any other supporting evidence.

3. Contact customer service: Reach out to our customer service team through our helpline or online chat. Provide them with details about your purchase and express your desire to request a rebate. They will guide you through the specific process for submitting your rebate request.

4. Submit the rebate request: Follow the instructions provided by customer service for submitting your rebate request. Make sure to attach all the required documentation and provide accurate and complete information.

5. Track your r

##### TF-IDF retrieval produced incorrect high-confidence matches for indirect queries such as "How do I get into my account?". This shows that lexical similarity alone is limited when customer phrasing does not contain strong intent-specific keywords. The system requires intent validation or semantic similarity in future versions.

In [37]:
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_df=0.85,
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(kb_df['clean_instruction'])

In [38]:
custom_stopwords = ['get', 'want', 'need', 'help', 'question']

def clean_user_query(query):
    query = query.lower()
    query = re.sub(r'[^a-zA-Z0-9\s]', '', query)
    
    words = query.split()
    words = [word for word in words if word not in custom_stopwords]
    
    return ' '.join(words)

In [41]:
user_query = input("Enter a new query:")
clean_query = clean_user_query(user_query)
query_vector = tfidf.transform([clean_query])
similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
)

best_match_index = similarity_scores.argmax()

best_match = kb_df.iloc[best_match_index]

print("Category:", best_match['category'])
print("Intent:", best_match['intent'])
print("Response:", best_match['response'])

Enter a new query: how do I get into my account


Category: ACCOUNT
Intent: create_account
Response: Thank you for considering opening another {{Account Type}} account with us! We appreciate your continued trust and loyalty to our services. To proceed, I kindly ask you to provide your full name, email address, and preferred username. Once I have these details, I'll guide you through the process of creating your new {{Account Type}} account seamlessly. Rest assured that we prioritize your security and privacy as our top concern.


## Query Enrichment

Some customer queries contain broad terms such as "account" without strong intent-specific keywords. This caused TF-IDF retrieval to confuse login-related issues with account creation or account management intents.

To improve retrieval quality, query enrichment was added. Instead of manually mapping every possible phrase, the system detects broader intent clues and adds stronger intent-related keywords before similarity matching.

In [42]:
def enrich_query(query):
    query = query.lower()
    query = re.sub(r'[^a-zA-Z0-9\s]', '', query)

    login_clues = ['into my account', 'access account', 'access my account', 'sign in', 'signin', 'login', 'log in', 'cant access', 'cannot access']
    
    if any(clue in query for clue in login_clues):
        query += ' login signin access password registration problems'

    return query

In [43]:
clean_query = enrich_query(user_query)

query_vector = tfidf.transform([clean_query])
similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

best_match_index = similarity_scores.argmax()
best_match = kb_df.iloc[best_match_index]

In [46]:
user_query = input("Enter a new query: ")

clean_query = enrich_query(user_query)

print("Clean Query:", clean_query)

query_vector = tfidf.transform([clean_query])

similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
).flatten()

best_match_index = similarity_scores.argmax()
best_match = kb_df.iloc[best_match_index]

print("Category:", best_match['category'])
print("Intent:", best_match['intent'])
print("Response:", best_match['response'])
print("Similarity Score:", similarity_scores[best_match_index])

Enter a new query:  How do I get into my account


Clean Query: how do i get into my account login signin access password registration problems
Category: ACCOUNT
Intent: registration_problems
Response: Thank you for contacting to us to inform about the registration problems you are encountering. We understand the importance of having a smooth signup process, and we apologize for any inconvenience caused. Your feedback is valuable to us as it helps us identify and address any issues in our registration system. Rest assured, we will investigate the problem and work towards finding a solution. If there are any specific details or error messages you can provide, it would greatly assist us in resolving the problem efficiently. Thank you for bringing this to our attention, and please know that we are committed to improving our registration process based on user feedback.
Similarity Score: 0.6042842591795284
